In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import sys
import asyncio
if sys.platform == "win32":
    if not isinstance(asyncio.get_event_loop_policy(),asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
        if "ipykernel" in sys.modules:
            sys.stderr=sys.__stderr__

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {              # Nickname for this server
            "transport": "streamable_http",        # HTTP transport (not stdio)
            "url": "http://127.0.0.1:8000/mcp"
        }
    }
)

In [4]:
tools = await client.get_tools()                           # All tools from all servers
resources = await client.get_resources("local_server")     # Read-only data from local_server
prompt = await client.get_prompt("local_server", "prompt") # Named prompt template
prompt = prompt[0].content

In [5]:
tools

[StructuredTool(name='search_web', description='Search the web using Tavily', args_schema={'properties': {'query': {'title': 'Query', 'type': 'string'}}, 'required': ['query'], 'title': 'search_webArguments', 'type': 'object'}, handle_tool_error=<function _handle_mcp_tool_error at 0x000001CF906579C0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x000001CF90684C20>)]

In [8]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    tools=tools,        # Tools sourced from MCP server
    system_prompt=prompt # Prompt sourced from MCP server
)

In [9]:
from langchain.messages import HumanMessage
config={"configurable":{"thread_id":"1"}}
response = await agent.ainvoke(
    {
        "messages": [HumanMessage(content="Can you get me latest updates of langchain agents?")]
    },
    config=config
)

In [10]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='Can you get me latest updates of langchain agents?', additional_kwargs={}, response_metadata={}, id='8f6b3ee2-3603-41fc-9461-26ef8b9f9bc9'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '8d09yxkfb', 'function': {'arguments': '{"query":"langchain agents latest updates"}', 'name': 'search_web'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 336, 'total_tokens': 354, 'completion_time': 0.026738945, 'completion_tokens_details': None, 'prompt_time': 0.020704255, 'prompt_tokens_details': None, 'queue_time': 0.159090713, 'total_time': 0.0474432}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f88bc-d7b6-7f43-a281-6879b5cade8d-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'langchain agents latest updates'}, 'id': '8